In [ ]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','xgboost==3.1.3','pandas==2.2.3','numpy==2.3.5','pyarrow==26.0.0','holidays==0.106'])


In [ ]:
from pathlib import Path
p=Path('/kaggle/working/march_run.py')
p.write_text('"""Two March full fits, original CUDA recipe and validation-selected hybrid history."""\nimport ast,gc,hashlib,io,json,platform,subprocess,time,warnings\nfrom pathlib import Path\nimport holidays\nimport numpy as np\nimport pandas as pd\nimport xgboost as xgb\nFUTURE_HYBRID_CSV=__import__(\'zlib\').decompress(__import__(\'base64\').b64decode(\'...\')).decode()\nEXPECTED_FULL_FEATURES={\'train\': {\'columns\': [\'days_lag_1\', \'is_quarter_end_month\', \'store_share_region_change_1_2\', \'is_year_start_month\', \'pto_lag_1_to_region__opening_date_category_pto_mean_lag_1\', \'pto_per_cash_register_hour_lag1\', \'pto_per_cash_register_lag1\', \'number_of_households\', \'pto_lag_1_to_roll_mean_6\', \'recent_mean_anchor_to_day_scaled_ratio\', \'log_daily_pto_diff_lag_1_2\', \'daily_growth_1_2_clipped\', \'pto_roll_cv_3\', \'pto_lag_1_to_roll_mean_3\', \'is_autumn\', \'holiday_month_type_cb\', \'season_cb\', \'population_size\', \'working_hours_per_day_lag_12_cat_cb\', \'avg_items_per_receipt_diff_lag_1_12\', \'log_pto_diff_lag_6_12\'], \'dtypes\': [\'float32\', \'int64\', \'float32\', \'int64\', \'float32\', \'float32\', \'float32\', \'int64\', \'float32\', \'float32\', \'float32\', \'float32\', \'float32\', \'float32\', \'int64\', \'float32\', \'float32\', \'int64\', \'float32\', \'float32\', \'float32\'], \'rows\': 485082, \'sha256\': \'34656309a5094e0e4145c5437458bfe50d32f6431f782f68c1b45c66f0674a8b\'}, \'test\': {\'columns\': [\'days_lag_1\', \'is_quarter_end_month\', \'store_share_region_change_1_2\', \'is_year_start_month\', \'pto_lag_1_to_region__opening_date_category_pto_mean_lag_1\', \'pto_per_cash_register_hour_lag1\', \'pto_per_cash_register_lag1\', \'number_of_households\', \'pto_lag_1_to_roll_mean_6\', \'recent_mean_anchor_to_day_scaled_ratio\', \'log_daily_pto_diff_lag_1_2\', \'daily_growth_1_2_clipped\', \'pto_roll_cv_3\', \'pto_lag_1_to_roll_mean_3\', \'is_autumn\', \'holiday_month_type_cb\', \'season_cb\', \'population_size\', \'working_hours_per_day_lag_12_cat_cb\', \'avg_items_per_receipt_diff_lag_1_12\', \'log_pto_diff_lag_6_12\'], \'dtypes\': [\'float32\', \'int64\', \'float32\', \'int64\', \'float32\', \'float32\', \'float32\', \'int64\', \'float32\', \'float32\', \'float32\', \'float32\', \'float32\', \'float32\', \'int64\', \'float32\', \'float32\', \'int64\', \'float32\', \'float32\', \'float32\'], \'rows\': 18657, \'sha256\': \'a1220e6a5806e7b03beefb7bb99899dbb1002b62bfa2dfd6ff8904c5f3f36899\'}, \'effective_target\': {\'columns\': [\'target\'], \'dtypes\': [\'float32\'], \'rows\': 485082, \'sha256\': \'e3a8931027d9cd3157423b11f7213de68c14124b30bf40b5c83a6582d15532f7\'}}\nEXPECTED_FUTURE_HYBRID_SHA256=\'b94bcbcdb387b1b2cbd54baad9adfc92890a321765af7d9e23f45ee1a61faef9\'\n\ndef fingerprint(frame):\n    return dict(columns=list(frame.columns),dtypes=[str(t) for t in frame.dtypes],rows=len(frame),\n        sha256=hashlib.sha256(pd.util.hash_pandas_object(frame,index=False,categorize=False).values.tobytes()).hexdigest())\n\ndef main():\n    assert xgb.__version__==\'3.1.3\'\n    original=list(Path(\'/kaggle/input\').rglob(\'input_manifest.json\'))\n    derived=list(Path(\'/kaggle/input\').rglob(\'research_manifest.json\'))\n    assert len(original)==len(derived)==1\n    source=original[0].parent;data=derived[0].parent;out=Path(\'/kaggle/working\')\n    manifest=json.loads(original[0].read_text());plan=json.loads(derived[0].read_text())\n    for name,digest in manifest[\'input_hashes\'].items():assert hashlib.sha256((source/name).read_bytes()).hexdigest()==digest\n    assert hashlib.sha256(FUTURE_HYBRID_CSV.encode()).hexdigest()==EXPECTED_FUTURE_HYBRID_SHA256\n    notebook=json.loads((source/\'original_solution.ipynb\').read_text());namespace={\'holidays\':holidays}\n    for i in [10,14,16,37]:exec(compile(\'\'.join(notebook[\'cells\'][i][\'source\']),\'original:\'+str(i),\'exec\'),namespace)\n    for node in ast.parse(\'\'.join(notebook[\'cells\'][18][\'source\'])).body:\n        if isinstance(node,ast.FunctionDef):exec(compile(ast.Module(body=[node],type_ignores=[]),\'original_preparation\',\'exec\'),namespace)\n        elif isinstance(node,ast.Assign) and isinstance(node.targets[0],ast.Name):\n            if node.targets[0].id in {\'COMMON_BASE_FEATURE_KWARGS\',\'ID_COL\',\'YEAR_COL\',\'MONTH_COL\',\'TIME_COL\',\'TARGET_COL\'}:\n                namespace[node.targets[0].id]=ast.literal_eval(node.value)\n    rename=ast.literal_eval(ast.parse(\'\'.join(notebook[\'cells\'][8][\'source\'])).body[0].value)\n    raw=pd.read_csv(source/\'train_2.csv\').rename(columns=rename)\n    with warnings.catch_warnings():\n        warnings.simplefilter(\'ignore\',pd.errors.PerformanceWarning)\n        features=namespace[\'build_retail_features_with_split\'](raw,split_mode=\'train_val_test\',min_year=2023,\n            val_year=2025,val_month=2,test_year=2025,test_month=3,create_test_if_missing=True,\n            expected_test_size=18657,use_val_as_test_history=True,\n            base_feature_kwargs=namespace[\'COMMON_BASE_FEATURE_KWARGS\'],verbose=True)\n    base=namespace[\'xgboost_log_diff_features\']\n    full=namespace[\'_prepare_numeric_model_pack\'](features,selected_features=base,target_mode=\'growth_log\',\n        label=\'xgboost_log_diff\',verbose=False)[\'test\']\n    train=full[\'X_train\'][base];test=full[\'X_test\'][base];target=np.asarray(full[\'y_train\'])\n    actual={\'train\':fingerprint(train),\'test\':fingerprint(test),\n        \'effective_target\':fingerprint(pd.DataFrame({\'target\':target}).astype(\'float32\').where(np.isfinite(target[:,None]),np.nan))}\n    assert actual==EXPECTED_FULL_FEATURES, \'Original full forecast matrices differ between local and GPU builds\'\n    train_meta=features[\'test_train_features\'][[\'new_id\',\'year_month_index\',\'pto\']].copy()\n    test_meta=features[\'test_features\'][[\'new_id\',\'year_month_index\',\'pto\',\'pto_lag_1\']].copy()\n    assert train_meta.year_month_index.max()==25 and (test_meta.year_month_index==26).all() and test_meta.pto.isna().all()\n    cols=plan[\'groups\'][\'hybrid8_history\']\n    historical=[]\n    for name in [\'2025-02_train.parquet\',\'2025-02_holdout.parquet\']:\n        assert hashlib.sha256((data/name).read_bytes()).hexdigest()==plan[\'hashes\'][name]\n        historical.append(pd.read_parquet(data/name,columns=[\'new_id\',\'year_month_index\']+cols))\n    extras=pd.concat(historical,ignore_index=True).set_index([\'new_id\',\'year_month_index\'])\n    future=pd.read_csv(io.StringIO(FUTURE_HYBRID_CSV)).set_index([\'new_id\',\'year_month_index\'])\n    for col in cols:future[col]=future[col].astype(\'float32\')\n    added_train=extras.reindex(pd.MultiIndex.from_frame(train_meta[[\'new_id\',\'year_month_index\']]))[cols].reset_index(drop=True).set_axis(train.index)\n    added_test=future.reindex(pd.MultiIndex.from_frame(test_meta[[\'new_id\',\'year_month_index\']]))[cols].reset_index(drop=True).set_axis(test.index)\n    assert not added_test.isna().any().any() and len(added_test)==18657\n    params=namespace[\'_clean_xgb_params\'](dict(namespace[\'xgboost_log_diff_params\']),keep_early_stopping=False)\n    assert params[\'n_estimators\']==3750 and params[\'device\']==\'cuda\' and \'early_stopping_rounds\' not in params\n    mask=np.isfinite(target);runtime=dict(python=platform.python_version(),\n        versions={name:__import__(\'importlib.metadata\',fromlist=[\'version\']).version(name) for name in [\'xgboost\',\'numpy\',\'pandas\',\'pyarrow\',\'holidays\']},\n        hardware=subprocess.check_output([\'nvidia-smi\'],text=True))\n    del features,raw,full;gc.collect()\n    records=[]\n    for variant in [\'baseline\',\'spatial\']:\n        xt=train if variant==\'baseline\' else pd.concat([train,added_train],axis=1)\n        xp=test if variant==\'baseline\' else pd.concat([test,added_test],axis=1)\n        start=time.perf_counter();print(\'FULL_CUDA_FIT\',variant,xt.shape,flush=True)\n        model=xgb.XGBRegressor(**params)\n        model.fit(xt.loc[mask],target[mask],verbose=False)\n        device=json.loads(model.get_booster().save_config())[\'learner\'][\'generic_param\'][\'device\']\n        assert device.startswith(\'cuda\') and model.get_booster().num_boosted_rounds()==3750\n        pred=namespace[\'inverse_forecast_target\'](model.predict(xp),test_meta,mode=\'growth_log\')\n        assert len(pred)==18657 and np.isfinite(pred).all() and (pred>=0).all()\n        directory=out/variant;directory.mkdir(exist_ok=True)\n        name=\'test_xgboost_log_diff\';table=pd.DataFrame({\'new_id\':test_meta.new_id.to_numpy(),\'rto\':pred})\n        table.to_csv(directory/(name+\'.csv\'),index=False);model.save_model(directory/(name+\'.ubj\'))\n        row=dict(model=\'xgboost_log_diff\',variant=variant,configuration=\'baseline\' if variant==\'baseline\' else \'hybrid8_history\',\n            forecast_month=\'2025-03\',training_months=[\'2023-01\',\'2025-02\'],train_rows=len(xt),fit_rows=int(mask.sum()),\n            features=list(xt.columns),parameters=params,trees=3750,early_stopping=False,\n            device=device,backend=\'CUDA\',versions=runtime[\'versions\'],seconds=time.perf_counter()-start,\n            notebook_sha256=manifest[\'input_hashes\'][\'original_solution.ipynb\'],raw_sha256=manifest[\'input_hashes\'][\'train_2.csv\'],\n            original_full_feature_checks=actual,future_hybrid_sha256=EXPECTED_FUTURE_HYBRID_SHA256 if variant==\'spatial\' else None,\n            validation_only=False,unknown_march_target=True,\n            submission_sha256=hashlib.sha256((directory/(name+\'.csv\')).read_bytes()).hexdigest(),\n            model_sha256=hashlib.sha256((directory/(name+\'.ubj\')).read_bytes()).hexdigest())\n        (directory/(name+\'.json\')).write_text(json.dumps(row,indent=2));records.append(row)\n        print(\'MARCH_CUDA_DONE\',variant,flush=True);del model;gc.collect()\n    (out/\'march_runtime.json\').write_text(json.dumps(runtime,indent=2))\n    (out/\'march_checks.json\').write_text(json.dumps(dict(status=\'passed\',runs=2,original_matrix_parity=True,\n        original_full_feature_checks=actual,forecast_month=\'2025-03\',unknown_march_target=True,trees_each=3750),indent=2))\n    (out/\'march_metrics.json\').write_text(json.dumps(records,indent=2))\n\nif __name__==\'__main__\':main()\n',encoding='utf-8')
subprocess.check_call([sys.executable,'-u',str(p)])
